In [ ]:
import json
import evaluate
from pathlib import Path
from tqdm.auto import tqdm
from seametrics.payload import Payload
from seametrics.payload.processor import PayloadProcessor

In [ ]:
MODELS = [
    "danv2n_sentry_2IR_1RGB_engine_c94af30_winter_yogurt_314_oversea_det",
    "danv2_IR640s_MIX1280n_cp_aug_engine_e7258f4_giddy_snow_337_oversea_det",
    "danv2_IR640s_MIX1280n_cp_aug_engine_e7258f4_swift_microwave_338_oversea_det",
]

ground_truth_fused = "ground_truth_det_fused"

AREA_RANGES_TUPLES = [
    ("all",    [0,      1e5**2]),
    ("small",  [0**2,   6**2  ]),
    ("medium", [6**2,   12**2 ]),
    ("large",  [12**2,  1e5**2]),
]

JSON_DIR    = Path(".")
OUTPUT_HTML = Path("benchmark_oversea.html")


In [ ]:
METRIC_KEYS = ["f1", "precision", "recall", "tp", "fp", "fn", "duplicates", "support", "fpi"]
AREAS       = ["all", "small", "medium", "large"]


def _short_name(model_name: str) -> str:
    parts = model_name.split("_")
    try:
        idx = parts.index("engine")
        suffix = "_".join(parts[idx + 1 : idx + 4])
        return "_".join(parts[:idx]) + f"_{suffix}"
    except ValueError:
        return model_name


def _json_path(model_name: str) -> Path:
    return JSON_DIR / f"metrics_{_short_name(model_name)}.json"


def fetch_payload(model_name: str, sequence_list: list = None):
    return PayloadProcessor(
        dataset_name="SENTRY_VIDEOS_DATASET_QA",
        tracking_mode=True,
        slices=["thermal_wide"],
        gt_field="ground_truth_det_fused",
        models=[model_name],
        tags=[],
        data_type="thermal",
        sequence_list=sequence_list,
    ).payload


def compute_and_save(model_name: str, payload, force: bool = False) -> dict:
    path = _json_path(model_name)
    if path.exists() and not force:
        print(f"Loading cached results for {_short_name(model_name)} ...")
        return json.loads(path.read_text())

    print(f"Computing overall metrics for {_short_name(model_name)} ({len(payload.sequences_list)} sequences) ...")
    overall_raw = evaluate.load(
        path="SEA-AI/det-metrics",
        iou_threshold=[0.00001],
        area_ranges_tuples=AREA_RANGES_TUPLES,
        payload=payload,
    ).compute()

    overall = {}
    for area in AREAS:
        m = overall_raw[model_name]["metrics"][area]
        overall[area] = {k: m[k] for k in METRIC_KEYS}

    per_seq = {}
    for seq_name in tqdm(payload.sequences_list, desc="Per-sequence"):
        seq_payload = Payload(
            dataset=payload.dataset,
            models=payload.models,
            gt_field_name=payload.gt_field_name,
            sequences={seq_name: payload.sequences[seq_name]},
        )
        res = evaluate.load(
            path="SEA-AI/det-metrics",
            iou_threshold=[0.00001],
            area_ranges_tuples=AREA_RANGES_TUPLES,
            payload=seq_payload,
        ).compute()
        per_seq[seq_name] = {}
        for area in AREAS:
            m = res[model_name]["metrics"][area]
            per_seq[seq_name][area] = {k: m[k] for k in METRIC_KEYS}

    result = {
        "model_name": model_name,
        "overall": overall,
        "per_sequence": per_seq,
    }
    path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(f"Saved → {path.resolve()}")
    return result


print("compute_and_save defined")

In [ ]:
import fiftyone as fo
from fiftyone import ViewField as F
from functools import reduce

# Only keep sequences where every model has at least one prediction in some frame
dataset = fo.load_dataset("SENTRY_VIDEOS_DATASET_QA")
frame_filter = reduce(
    lambda a, b: a & b,
    [F(f"{model}.detections").length() > 0 for model in MODELS] +
    [F(f"{ground_truth_fused}.detections").length() > 0]
)
view = dataset.select_group_slices("thermal_wide").match_frames(frame_filter)
sequence_list = view.distinct("sequence")
print(f"Sequences with predictions for all models: {len(sequence_list)}")

# Compute metrics for every model on the same sequence list
all_results = {}
for model in MODELS:
    if _json_path(model).exists():
        all_results[model] = compute_and_save(model, payload=None)
    else:
        payload = fetch_payload(model, sequence_list=sequence_list)
        all_results[model] = compute_and_save(model, payload)

print("All models done.")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ── config ───────────────────────────────────────────────────
MODEL_A = list(all_results.keys())[0]
MODEL_B = list(all_results.keys())[1]
AREA    = "all"   # "all" | "small" | "medium" | "large"

AREAS       = ["all", "small", "medium", "large"]
AREA_LABELS = ["All", "Small", "Medium", "Large"]
COLORS      = ["#0d6efd", "#ffc107"]

sn_a = _short_name(MODEL_A)
sn_b = _short_name(MODEL_B)

# ── bar charts ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Performance by Area Range", fontsize=14, fontweight="bold", y=1.01)

x = np.arange(len(AREA_LABELS))
w = 0.35

for ax, metric, title in zip(axes, ["f1", "precision", "recall"], ["F1", "Precision", "Recall"]):
    va = [all_results[MODEL_A]["overall"][a][metric] * 100 for a in AREAS]
    vb = [all_results[MODEL_B]["overall"][a][metric] * 100 for a in AREAS]

    bars_a = ax.bar(x - w/2, va, w, color=COLORS[0], label=sn_a)
    bars_b = ax.bar(x + w/2, vb, w, color=COLORS[1], label=sn_b)

    for i, (a_val, b_val, bar) in enumerate(zip(va, vb, bars_b)):
        diff  = b_val - a_val
        arrow = " ↑" if diff > 0.05 else (" ↓" if diff < -0.05 else "")
        color = "#198754" if diff > 0.05 else ("#dc3545" if diff < -0.05 else "#333")
        ax.text(bar.get_x() + bar.get_width()/2, b_val + 0.8,
                f"{b_val:.1f}%{arrow}", ha="center", va="bottom",
                fontsize=7.5, fontweight="bold", color=color)
    for bar, v in zip(bars_a, va):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.8, f"{v:.1f}%",
                ha="center", va="bottom", fontsize=7.5, color="#555")

    ax.set_title(title, fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(AREA_LABELS)
    ax.set_ylim(0, 110)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.legend(fontsize=7.5, loc="lower right")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:

import urllib.request as _ur
from pathlib import Path as _P

def _fetch_js(url: str, cache_path: str) -> str:
    p = _P(cache_path)
    if not p.exists():
        _ur.urlretrieve(url, cache_path)
    return p.read_text(encoding="utf-8")

_CHARTJS    = _fetch_js("https://cdn.jsdelivr.net/npm/chart.js@4.4.2/dist/chart.umd.min.js", "/tmp/chart.umd.min.js")
_DATALABELS = _fetch_js("https://cdn.jsdelivr.net/npm/chartjs-plugin-datalabels@2.2.0/dist/chartjs-plugin-datalabels.min.js", "/tmp/chartjs-plugin-datalabels.min.js")

def build_html(all_results: dict) -> str:
    all_models_js = {}
    for model_name, data in all_results.items():
        all_models_js[model_name] = {
            "short_name": _short_name(model_name),
            "overall":      data["overall"],
            "per_sequence": data["per_sequence"],
        }

    all_models_json = json.dumps(all_models_js)
    model_keys      = list(all_results.keys())
    default_a_json  = json.dumps(model_keys[0])
    default_b_json  = json.dumps(model_keys[1])

    return f"""\
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Model Comparison</title>
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.2/dist/css/bootstrap.min.css">
<script>{_CHARTJS}<\/script>
<script>{_DATALABELS}<\/script>
<style>
body {{ background:#f4f6f9; font-size:0.9rem; }}
.card {{ box-shadow:0 2px 8px rgba(0,0,0,.08); border:none; border-radius:12px; }}
.card-header {{ background:#fff; border-bottom:1px solid #e9ecef; border-radius:12px 12px 0 0 !important; font-weight:600; font-size:1rem; }}
th {{ cursor:pointer; user-select:none; white-space:nowrap; }}
th:hover {{ background:#e9ecef !important; }}
.sort-icon {{ opacity:.35; font-size:.7rem; margin-left:3px; }}
.sort-asc .sort-icon::after  {{ content:"▲"; opacity:1; }}
.sort-desc .sort-icon::after {{ content:"▼"; opacity:1; }}
.sort-icon::after {{ content:"⇅"; }}
.diff-pos {{ color:#198754; font-weight:600; }}
.diff-neg {{ color:#dc3545; font-weight:600; }}
.diff-zero {{ color:#adb5bd; }}
.ma-badge {{ background:#0d6efd; color:#fff; }}
.mb-badge {{ background:#ffc107; color:#212529; }}
.metric-lbl {{ font-size:.78rem; text-transform:uppercase; letter-spacing:.04em; color:#6c757d; font-weight:600; }}
#seq-search {{ max-width:280px; }}
.area-tabs .nav-link {{ padding:.3rem .8rem; }}
.seq-mono {{ font-family:monospace; font-size:.82rem; }}
.hdr-ma {{ background:#e7f0ff; }}
.hdr-mb {{ background:#fff8e0; }}
.hdr-diff {{ background:#f0f0f0; }}
.model-selector {{ min-width:260px; }}
</style>
</head>
<body>
<div class="container-xl py-4">

  <!-- ===== MODEL SELECTOR ===== -->
  <div class="card mb-4">
    <div class="card-body">
      <div class="row align-items-center g-3">
        <div class="col-auto fw-semibold">Compare:</div>
        <div class="col-auto">
          <label class="form-label mb-1 small text-muted">Model A</label>
          <select class="form-select model-selector" id="sel-model-a"></select>
        </div>
        <div class="col-auto fw-bold text-secondary fs-5">vs</div>
        <div class="col-auto">
          <label class="form-label mb-1 small text-muted">Model B</label>
          <select class="form-select model-selector" id="sel-model-b"></select>
        </div>
        <div class="col-auto ms-2">
          <span class="text-secondary small">&#916; = B &minus; A</span>
        </div>
      </div>
    </div>
  </div>

  <!-- ===== PART 1: BAR CHARTS ===== -->
  <div class="card mb-4">
    <div class="card-header">Performance by Area Range</div>
    <div class="card-body">
      <div class="row g-4">
        <div class="col-4"><canvas id="chart-f1"></canvas></div>
        <div class="col-4"><canvas id="chart-precision"></canvas></div>
        <div class="col-4"><canvas id="chart-recall"></canvas></div>
      </div>
    </div>
  </div>

  <!-- ===== PART 2: OVERALL TABLE ===== -->
  <div class="card mb-4">
    <div class="card-header d-flex align-items-center gap-3">
      Overall Metrics
      <ul class="nav nav-pills area-tabs ms-auto" id="overall-tabs">
        <li class="nav-item"><a class="nav-link active" href="#" data-area="all">All</a></li>
        <li class="nav-item"><a class="nav-link" href="#" data-area="small">Small</a></li>
        <li class="nav-item"><a class="nav-link" href="#" data-area="medium">Medium</a></li>
        <li class="nav-item"><a class="nav-link" href="#" data-area="large">Large</a></li>
      </ul>
    </div>
    <div class="card-body p-0">
      <table class="table table-hover mb-0" id="overall-table">
        <thead class="table-light">
          <tr>
            <th style="width:140px">Metric</th>
            <th class="text-end hdr-ma" id="overall-th-ma"></th>
            <th class="text-end hdr-mb" id="overall-th-mb"></th>
            <th class="text-end hdr-diff">&#916; (B &minus; A)</th>
          </tr>
        </thead>
        <tbody id="overall-tbody"></tbody>
      </table>
    </div>
  </div>

  <!-- ===== PART 3: PER-SEQUENCE ===== -->
  <div class="card">
    <div class="card-header d-flex flex-wrap align-items-center gap-3">
      Per-Sequence Metrics
      <ul class="nav nav-pills area-tabs" id="seq-tabs">
        <li class="nav-item"><a class="nav-link active" href="#" data-area="all">All</a></li>
        <li class="nav-item"><a class="nav-link" href="#" data-area="small">Small</a></li>
        <li class="nav-item"><a class="nav-link" href="#" data-area="medium">Medium</a></li>
        <li class="nav-item"><a class="nav-link" href="#" data-area="large">Large</a></li>
      </ul>
      <div class="ms-auto d-flex gap-2 align-items-center">
        <input type="text" class="form-control form-control-sm" id="seq-search" placeholder="Filter sequences...">
        <select class="form-select form-select-sm" id="metric-select" style="width:auto">
          <option value="f1">F1</option>
          <option value="precision">Precision</option>
          <option value="recall">Recall</option>
          <option value="tp">TP</option>
          <option value="fp">FP</option>
          <option value="fn">FN</option>
          <option value="duplicates">Duplicates</option>
          <option value="fpi">FPI</option>
        </select>
      </div>
    </div>
    <div class="card-body p-0">
      <div class="table-responsive">
        <table class="table table-hover table-sm mb-0" id="seq-table">
          <thead class="table-light">
            <tr>
              <th rowspan="2" class="align-middle" data-col="sequence" style="min-width:220px">
                Sequence <span class="sort-icon"></span>
              </th>
              <th colspan="3" class="text-center hdr-ma" id="seq-th-ma"></th>
              <th colspan="3" class="text-center hdr-mb" id="seq-th-mb"></th>
              <th colspan="3" class="text-center hdr-diff">&#916; (B &minus; A)</th>
            </tr>
            <tr id="seq-subheader"></tr>
          </thead>
          <tbody id="seq-tbody"></tbody>
        </table>
      </div>
      <div class="text-muted small p-2" id="seq-count"></div>
    </div>
  </div>

</div>
<script>
Chart.register(ChartDataLabels);

const ALL_MODELS = {all_models_json};

const METRIC_CFG = {{
  f1:         {{ label:"F1",         fmt:"pct", hib:true  }},
  precision:  {{ label:"Precision",  fmt:"pct", hib:true  }},
  recall:     {{ label:"Recall",     fmt:"pct", hib:true  }},
  tp:         {{ label:"TP",         fmt:"int", hib:true  }},
  fp:         {{ label:"FP",         fmt:"int", hib:false }},
  fn:         {{ label:"FN",         fmt:"int", hib:false }},
  duplicates: {{ label:"Duplicates", fmt:"int", hib:false }},
  support:    {{ label:"Support",    fmt:"int", hib:null  }},
  fpi:        {{ label:"FPI",        fmt:"int", hib:false }},
}};

const AREAS       = ["all", "small", "medium", "large"];
const AREA_LABELS = ["All", "Small", "Medium", "Large"];

let modelAKey = {default_a_json};
let modelBKey = {default_b_json};
let overallArea = "all";
let seqArea     = "all";
let seqFilter   = "";
let seqMetric   = "f1";
let sortCol     = null;
let sortDir     = 1;
let chartInstances = {{}};

function fmtPct(v)  {{ return (v * 100).toFixed(1) + "%"; }}
function fmtInt(v)  {{ return Number(v).toLocaleString(); }}
function fmt(v, f)  {{ return f === "pct" ? fmtPct(v) : fmtInt(v); }}
function diffClass(d, hib) {{
  if (hib === null || Math.abs(d) < 1e-9) return "diff-zero";
  return (d > 0) === hib ? "diff-pos" : "diff-neg";
}}
function fmtDiff(d, f, hib) {{
  const sign = d > 0 ? "+" : "";
  return `<span class="${{diffClass(d, hib)}}">${{sign}}${{fmt(d, f)}}</span>`;
}}

function getMA() {{ return ALL_MODELS[modelAKey]; }}
function getMB() {{ return ALL_MODELS[modelBKey]; }}

// ── model selectors ─────────────────────────────────────────
function initSelectors() {{
  const selA = document.getElementById("sel-model-a");
  const selB = document.getElementById("sel-model-b");
  Object.entries(ALL_MODELS).forEach(([key, data]) => {{
    selA.add(new Option(data.short_name, key));
    selB.add(new Option(data.short_name, key));
  }});
  selA.value = modelAKey;
  selB.value = modelBKey;
  selA.addEventListener("change", e => {{ modelAKey = e.target.value; renderAll(); }});
  selB.addEventListener("change", e => {{ modelBKey = e.target.value; renderAll(); }});
}}

function updateBadges() {{
  const ma = getMA().short_name;
  const mb = getMB().short_name;
  document.getElementById("overall-th-ma").innerHTML = `<span class="badge ma-badge">${{ma}}</span>`;
  document.getElementById("overall-th-mb").innerHTML = `<span class="badge mb-badge">${{mb}}</span>`;
  document.getElementById("seq-th-ma").innerHTML = `<span class="badge ma-badge">${{ma}}</span>`;
  document.getElementById("seq-th-mb").innerHTML = `<span class="badge mb-badge">${{mb}}</span>`;
}}

// ── bar charts ───────────────────────────────────────────────
function renderCharts() {{
  const ma = getMA(), mb = getMB();
  const chartDefs = [
    {{ id:"chart-f1",        metricKey:"f1",        title:"F1" }},
    {{ id:"chart-precision", metricKey:"precision", title:"Precision" }},
    {{ id:"chart-recall",    metricKey:"recall",    title:"Recall" }},
  ];
  chartDefs.forEach(def => {{
    const dataA = AREAS.map(a => +(ma.overall[a][def.metricKey] * 100).toFixed(2));
    const dataB = AREAS.map(a => +(mb.overall[a][def.metricKey] * 100).toFixed(2));
    const base = {{ anchor:"end", align:"top", font:{{ size:9, weight:"bold" }} }};
    if (chartInstances[def.id]) chartInstances[def.id].destroy();
    chartInstances[def.id] = new Chart(document.getElementById(def.id), {{
      type: "bar",
      data: {{
        labels: AREA_LABELS,
        datasets: [
          {{
            label: ma.short_name, data: dataA, backgroundColor: "#0d6efd",
            datalabels: {{ ...base, color: "#333", formatter: v => v.toFixed(1) + "%" }},
          }},
          {{
            label: mb.short_name, data: dataB, backgroundColor: "#ffc107",
            datalabels: {{
              ...base,
              formatter: (v, ctx) => {{
                const diff = v - dataA[ctx.dataIndex];
                const arrow = diff > 0.05 ? " ↑" : diff < -0.05 ? " ↓" : "";
                return v.toFixed(1) + "%" + arrow;
              }},
              color: (ctx) => {{
                const diff = dataB[ctx.dataIndex] - dataA[ctx.dataIndex];
                return diff > 0.05 ? "#198754" : diff < -0.05 ? "#dc3545" : "#333";
              }},
            }},
          }},
        ],
      }},
      options: {{
        responsive: true,
        layout: {{ padding: {{ top: 16 }} }},
        plugins: {{
          title:  {{ display: true, text: def.title, font: {{ size: 14 }} }},
          legend: {{ position: "bottom", labels: {{ boxWidth: 12 }} }},
          tooltip: {{ callbacks: {{ label: ctx => `${{ctx.dataset.label}}: ${{ctx.parsed.y.toFixed(1)}}%` }} }},
        }},
        scales: {{
          y: {{
            beginAtZero: true, max: 100,
            ticks: {{ callback: v => v + "%" }},
          }},
        }},
      }},
    }});
  }});
}}

// ── overall table ────────────────────────────────────────────
function renderOverall() {{
  const da = getMA().overall[overallArea];
  const db = getMB().overall[overallArea];
  const rows = Object.entries(METRIC_CFG).map(([key, cfg]) => {{
    const av = da[key], bv = db[key], d = bv - av;
    return `<tr>
      <td class="metric-lbl">${{cfg.label}}</td>
      <td class="text-end hdr-ma">${{fmt(av, cfg.fmt)}}</td>
      <td class="text-end hdr-mb">${{fmt(bv, cfg.fmt)}}</td>
      <td class="text-end hdr-diff">${{fmtDiff(d, cfg.fmt, cfg.hib)}}</td>
    </tr>`;
  }});
  document.getElementById("overall-tbody").innerHTML = rows.join("");
}}

document.querySelectorAll("#overall-tabs .nav-link").forEach(a => {{
  a.addEventListener("click", e => {{
    e.preventDefault();
    document.querySelectorAll("#overall-tabs .nav-link").forEach(x => x.classList.remove("active"));
    a.classList.add("active");
    overallArea = a.dataset.area;
    renderOverall();
  }});
}});

// ── per-sequence table ────────────────────────────────────────
function buildSubHeader() {{
  const lbl = METRIC_CFG[seqMetric].label;
  const cols = [
    {{ key:"ma_metric", cls:"hdr-ma",   label:lbl       }},
    {{ key:"ma_tp",     cls:"hdr-ma",   label:"TP"       }},
    {{ key:"ma_fp",     cls:"hdr-ma",   label:"FP"       }},
    {{ key:"mb_metric", cls:"hdr-mb",   label:lbl       }},
    {{ key:"mb_tp",     cls:"hdr-mb",   label:"TP"       }},
    {{ key:"mb_fp",     cls:"hdr-mb",   label:"FP"       }},
    {{ key:"d_metric",  cls:"hdr-diff", label:"&#916;"+lbl }},
    {{ key:"d_tp",      cls:"hdr-diff", label:"&#916;TP" }},
    {{ key:"d_fp",      cls:"hdr-diff", label:"&#916;FP" }},
  ];
  document.getElementById("seq-subheader").innerHTML = cols.map(c =>
    `<th class="text-end ${{c.cls}}" data-col="${{c.key}}">
       ${{c.label}} <span class="sort-icon"></span>
     </th>`
  ).join("");
  document.querySelectorAll("#seq-table th[data-col]").forEach(th => {{
    th.addEventListener("click", () => {{
      const col = th.dataset.col;
      if (sortCol === col) {{
        sortDir *= -1;
        th.classList.toggle("sort-asc");
        th.classList.toggle("sort-desc");
      }} else {{
        document.querySelectorAll("#seq-table th").forEach(t =>
          t.classList.remove("sort-asc", "sort-desc"));
        sortCol = col;
        sortDir = 1;
        th.classList.add("sort-asc");
      }}
      renderSeq();
    }});
  }});
}}

function getRow(seqName) {{
  const sa = getMA().per_sequence[seqName][seqArea];
  const sb = getMB().per_sequence[seqName][seqArea];
  return {{
    sequence:  seqName,
    ma_metric: sa[seqMetric], ma_tp: sa.tp, ma_fp: sa.fp,
    mb_metric: sb[seqMetric], mb_tp: sb.tp, mb_fp: sb.fp,
    d_metric:  sb[seqMetric] - sa[seqMetric],
    d_tp:      sb.tp - sa.tp,
    d_fp:      sb.fp - sa.fp,
  }};
}}

function renderSeq() {{
  const cfg = METRIC_CFG[seqMetric];
  const allSeqs = Object.keys(getMA().per_sequence);
  let rows = allSeqs.map(getRow);
  if (seqFilter) {{
    const q = seqFilter.toLowerCase();
    rows = rows.filter(r => r.sequence.toLowerCase().includes(q));
  }}
  if (sortCol) {{
    rows.sort((a, b) => {{
      const av = a[sortCol], bv = b[sortCol];
      return typeof av === "string" ? sortDir * av.localeCompare(bv) : sortDir * (av - bv);
    }});
  }}
  document.getElementById("seq-tbody").innerHTML = rows.map(r => `<tr>
    <td class="seq-mono">${{r.sequence}}</td>
    <td class="text-end hdr-ma">${{fmt(r.ma_metric, cfg.fmt)}}</td>
    <td class="text-end hdr-ma">${{fmtInt(r.ma_tp)}}</td>
    <td class="text-end hdr-ma">${{fmtInt(r.ma_fp)}}</td>
    <td class="text-end hdr-mb">${{fmt(r.mb_metric, cfg.fmt)}}</td>
    <td class="text-end hdr-mb">${{fmtInt(r.mb_tp)}}</td>
    <td class="text-end hdr-mb">${{fmtInt(r.mb_fp)}}</td>
    <td class="text-end hdr-diff">${{fmtDiff(r.d_metric, cfg.fmt, cfg.hib)}}</td>
    <td class="text-end hdr-diff">${{fmtDiff(r.d_tp,     "int",   true  )}}</td>
    <td class="text-end hdr-diff">${{fmtDiff(r.d_fp,     "int",   false )}}</td>
  </tr>`).join("");
  document.getElementById("seq-count").textContent =
    `Showing ${{rows.length}} of ${{allSeqs.length}} sequences`;
}}

document.querySelectorAll("#seq-tabs .nav-link").forEach(a => {{
  a.addEventListener("click", e => {{
    e.preventDefault();
    document.querySelectorAll("#seq-tabs .nav-link").forEach(x => x.classList.remove("active"));
    a.classList.add("active");
    seqArea = a.dataset.area;
    renderSeq();
  }});
}});

document.getElementById("seq-search").addEventListener("input", e => {{
  seqFilter = e.target.value;
  renderSeq();
}});

document.getElementById("metric-select").addEventListener("change", e => {{
  seqMetric = e.target.value;
  buildSubHeader();
  renderSeq();
}});

function renderAll() {{
  updateBadges();
  renderCharts();
  renderOverall();
  buildSubHeader();
  renderSeq();
}}

initSelectors();
renderAll();
</script>
</body>
</html>"""


print("build_html defined")


In [ ]:
html = build_html(all_results)
OUTPUT_HTML.write_text(html, encoding="utf-8")
print(f"Saved → {OUTPUT_HTML.resolve()}")
